In [4]:
import os
import pandas as pd
import numpy as np


CONTINUOUS_FEATURES = [

    'LapTime_Seconds',
    'Position',
    'LapNumber',
    'Stint',
    'TyreLife',
    'TrackStatus',

    'delta_laptime',
    'CumulativeTimeStint',
    'race_progress_fraction',

    'relative_tire_age',
    'tire_compound_age_delta',
    'tire_performance_decay',
    'rolling_pace_mean_5',
    'pace_std_5',
    'pace_degradation_slope',

    'historical_pit_lap',
    'pit_window_delta',

    'Speed_mean',
    'Speed_max',
    'Speed_std',

    'RPM_mean',
    'RPM_max',

    'Throttle_mean',
    'Throttle_std',

    'Brake_mean',
    'Brake_sum',

    'DRS_mean',
    'DRS_sum',

    'DistanceToDriverAhead_mean',
    'DistanceToDriverAhead_min',

    'nGear_mean',
    'nGear_max',

    'traffic_pressure',
    'drs_dependency',
    'thermal_stress_proxy',
    'high_speed_stress',
    'brake_aggression',
    'pace_vs_ahead'
]


CATEGORICAL_FEATURES = [
    'Compound',
    'Team'
]


METADATA_COLUMNS = [
    'Year',
    'RaceID',
    'Driver',
    'DriverNumber'
]


TARGET_COLUMN = [
    'HasPitStop'
]


def load_processed_files(processed_root):

    all_dfs = []

    for year in os.listdir(processed_root):

        year_path = os.path.join(
            processed_root,
            year
        )

        if not os.path.isdir(year_path):
            continue

        for file in os.listdir(year_path):

            if not file.endswith("_processed.csv"):
                continue

            file_path = os.path.join(
                year_path,
                file
            )

            try:

                df = pd.read_csv(
                    file_path
                )

                race_id = file.replace(
                    "_processed.csv",
                    ""
                )

                df['Year'] = int(year)

                df['RaceID'] = race_id

                all_dfs.append(df)

                print(
                    f"Loaded {year} {race_id} "
                    f"({len(df)} rows)"
                )

            except Exception as e:

                print(
                    f"FAILED {file}: {e}"
                )

    if len(all_dfs) == 0:
        raise ValueError(
            "No processed files found"
        )

    master_df = pd.concat(
        all_dfs,
        ignore_index=True
    )

    return master_df


def select_training_columns(df):

    required_columns = (
        METADATA_COLUMNS +
        CONTINUOUS_FEATURES +
        CATEGORICAL_FEATURES +
        TARGET_COLUMN
    )

    existing_columns = [
        col for col in required_columns
        if col in df.columns
    ]

    missing_columns = [
        col for col in required_columns
        if col not in df.columns
    ]

    if len(missing_columns) > 0:

        print("\nMissing Columns:")
        for col in missing_columns:
            print(col)

    df = df[
        existing_columns
    ].copy()

    return df


def clean_training_dataframe(df):

    numeric_cols = df.select_dtypes(
        include=np.number
    ).columns

    df[numeric_cols] = (
        df[numeric_cols]
        .replace(
            [np.inf, -np.inf],
            np.nan
        )
    )

    df[numeric_cols] = (
        df[numeric_cols]
        .fillna(0)
    )

    categorical_cols = [
        col for col in CATEGORICAL_FEATURES
        if col in df.columns
    ]

    for col in categorical_cols:

        df[col] = (
            df[col]
            .astype(str)
            .fillna("UNKNOWN")
        )

    return df


def save_master_training_csv(
    processed_root,
    output_csv="all_training_data.csv"
):

    print("\nLoading processed race files...")

    df = load_processed_files(
        processed_root
    )

    print("\nSelecting features...")

    df = select_training_columns(
        df
    )

    print("\nCleaning dataframe...")

    df = clean_training_dataframe(
        df
    )

    print("\nDataset Summary")
    print(f"Rows: {len(df)}")
    print(f"Columns: {len(df.columns)}")

    print("\nYears:")
    print(
        sorted(
            df['Year'].unique()
        )
    )

    print("\nPit Stop Distribution:")
    print(
        df['HasPitStop']
        .value_counts()
    )

    df.to_csv(
        output_csv,
        index=False
    )

    print(
        f"\nSaved training dataset to:\n"
        f"{output_csv}"
    )

    return df


if __name__ == "__main__":

    PROCESSED_ROOT = (
        r"C:/Nguyen Tri/Code/Statisanalyss/Preprocessed"
    )

    OUTPUT_CSV = (
        r"C:/Nguyen Tri/Code/Statisanalyss/all_training_data.csv"
    )

    df = save_master_training_csv(
        processed_root=PROCESSED_ROOT,
        output_csv=OUTPUT_CSV
    )

    print("\nDONE")


Loading processed race files...
Loaded 2018 Abu Dhabi Grand Prix (0 rows)
Loaded 2018 Austrian Grand Prix (791 rows)
Loaded 2018 Azerbaijan Grand Prix (294 rows)
Loaded 2018 Belgian Grand Prix (329 rows)
Loaded 2018 Brazilian Grand Prix (921 rows)
Loaded 2018 British Grand Prix (881 rows)
Loaded 2018 Canadian Grand Prix (0 rows)
Loaded 2018 Chinese Grand Prix (748 rows)
Loaded 2018 French Grand Prix (358 rows)
Loaded 2018 German Grand Prix (749 rows)
Loaded 2018 Hungarian Grand Prix (893 rows)
Loaded 2018 Japanese Grand Prix (765 rows)
Loaded 2018 Mexican Grand Prix (0 rows)
Loaded 2018 Monaco Grand Prix (0 rows)
Loaded 2018 Russian Grand Prix (715 rows)
Loaded 2018 Singapore Grand Prix (397 rows)
Loaded 2018 Spanish Grand Prix (961 rows)
Loaded 2018 United States Grand Prix (501 rows)
Loaded 2019 Abu Dhabi Grand Prix (1075 rows)
Loaded 2019 Australian Grand Prix (1041 rows)
Loaded 2019 Austrian Grand Prix (1401 rows)
Loaded 2019 Azerbaijan Grand Prix (948 rows)
Loaded 2019 Bahrain Gr

C:\Users\PC\AppData\Local\Temp\ipykernel_20424\3134839703.py:136: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  master_df = pd.concat(



Selecting features...

Cleaning dataframe...

Dataset Summary
Rows: 152475
Columns: 45

Years:
[np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]

Pit Stop Distribution:
HasPitStop
0    147657
1      4818
Name: count, dtype: int64

Saved training dataset to:
C:/Nguyen Tri/Code/Statisanalyss/all_training_data.csv

DONE


In [5]:
print(df.head())
df.describe()

   Year               RaceID Driver DriverNumber  LapTime_Seconds  Position  \
0  2018  Austrian Grand Prix    ALO           14           87.388      17.0   
1  2018  Austrian Grand Prix    ALO           14           69.835      17.0   
2  2018  Austrian Grand Prix    ALO           14           70.238      17.0   
3  2018  Austrian Grand Prix    ALO           14           70.354      17.0   
4  2018  Austrian Grand Prix    ALO           14           70.125      17.0   

   LapNumber  Stint  TyreLife  TrackStatus  ...  nGear_max  traffic_pressure  \
0       16.0    2.0       1.0         71.0  ...        8.0          0.001613   
1       17.0    2.0       2.0          1.0  ...        8.0          0.001434   
2       18.0    2.0       3.0          1.0  ...        8.0          0.001403   
3       19.0    2.0       4.0          1.0  ...        8.0          0.001367   
4       20.0    2.0       5.0          1.0  ...        8.0          0.001362   

   drs_dependency  thermal_stress_proxy  hig

,Year,LapTime_Seconds,Position,LapNumber,Stint,TyreLife,TrackStatus,delta_laptime,CumulativeTimeStint,race_progress_fraction,...,DistanceToDriverAhead_mean,DistanceToDriverAhead_min,nGear_mean,nGear_max,traffic_pressure,drs_dependency,thermal_stress_proxy,high_speed_stress,brake_aggression,pace_vs_ahead
count,152475.000000,152475.000000,152475.000000,152475.000000,152475.000000,152475.000000,152475.000000,152475.000000,152475.000000,152475.000000,...,152475.000000,152475.000000,152475.000000,152475.000000,152475.000000,152475.000000,1.524750e+05,152475.000000,152475.000000,152475.000000
mean,2021.856954,89.645814,9.800846,31.051471,2.032399,15.143965,15.540856,-0.299531,1280.514121,0.504180,...,296.180100,129.524359,5.061855,7.831133,0.025133,5.603816,6.009870e+05,12556.644479,27989.261601,4.988154
std,2.215135,30.115262,5.451492,18.222725,0.920876,10.641102,586.532276,19.513392,924.712744,0.284748,...,768.133399,675.855369,0.749966,0.995827,0.080951,112.713917,3.354821e+05,3139.667581,8553.780075,16.508729
min,2018.000000,0.000000,0.000000,1.000000,1.000000,0.000000,1.000000,-2414.122000,0.000000,0.011494,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000
25%,2020.000000,80.454500,5.000000,16.000000,1.000000,7.000000,1.000000,-0.303000,549.237500,0.258621,...,66.310091,0.180000,4.803112,8.000000,0.003205,0.000000,4.852164e+05,11077.925480,23986.641000,0.630414
50%,2022.000000,88.233000,10.000000,30.000000,2.000000,13.000000,1.000000,0.000000,1108.624000,0.500000,...,141.865182,0.884444,5.165498,8.000000,0.006964,0.000000,5.744897e+05,12870.489515,27324.338557,1.374154
75%,2024.000000,99.109500,14.000000,45.000000,3.000000,21.000000,1.000000,0.280000,1814.943500,0.750000,...,308.840892,46.716944,5.527589,8.000000,0.014765,4.046596,6.861277e+05,14586.257120,31233.471996,2.877515
max,2025.000000,2526.253000,20.000000,87.000000,7.000000,78.000000,67124.000000,71.028000,6158.100000,1.000000,...,149179.430030,147483.182779,39.207407,128.000000,0.987253,8651.586424,5.628041e+07,27324.545161,817631.997964,225.269432


In [6]:
!pip install tensorflow

In [7]:
import torch
import torch.nn as nn


class F1PitStopPredictor(nn.Module):

    def __init__(
        self,
        input_size,
        hidden1=256,
        hidden2=128,
        hidden3=64,
        dropout=0.3
    ):

        super(F1PitStopPredictor, self).__init__()

        # =====================================================
        # BiLSTM 1
        # =====================================================

        self.lstm1 = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden1,
            num_layers=1,
            batch_first=True,
            bidirectional=True,
            dropout=0
        )

        self.bn1 = nn.BatchNorm1d(
            hidden1 * 2
        )

        self.dropout1 = nn.Dropout(
            0.2
        )

        # =====================================================
        # BiLSTM 2
        # =====================================================

        self.lstm2 = nn.LSTM(
            input_size=hidden1 * 2,
            hidden_size=hidden2,
            num_layers=1,
            batch_first=True,
            bidirectional=True,
            dropout=0
        )

        self.bn2 = nn.BatchNorm1d(
            hidden2 * 2
        )

        self.dropout2 = nn.Dropout(
            0.3
        )

        # =====================================================
        # BiLSTM 3
        # =====================================================

        self.lstm3 = nn.LSTM(
            input_size=hidden2 * 2,
            hidden_size=hidden3,
            num_layers=1,
            batch_first=True,
            bidirectional=True,
            dropout=0
        )

        self.bn3 = nn.BatchNorm1d(
            hidden3 * 2
        )

        self.dropout3 = nn.Dropout(
            0.3
        )

        # =====================================================
        # Dense Layers
        # =====================================================

        self.fc1 = nn.Linear(
            hidden3 * 2,
            64
        )

        self.dropout4 = nn.Dropout(
            0.2
        )

        self.fc2 = nn.Linear(
            64,
            32
        )

        self.output = nn.Linear(
            32,
            1
        )

        self.relu = nn.ReLU()

        self.sigmoid = nn.Sigmoid()

    def forward(self, x):

        # =====================================================
        # LSTM 1
        # =====================================================

        x, _ = self.lstm1(x)

        x = self.dropout1(x)

        x = x.permute(0, 2, 1)
        x = self.bn1(x)
        x = x.permute(0, 2, 1)

        # =====================================================
        # LSTM 2
        # =====================================================

        x, _ = self.lstm2(x)

        x = self.dropout2(x)

        x = x.permute(0, 2, 1)
        x = self.bn2(x)
        x = x.permute(0, 2, 1)

        # =====================================================
        # LSTM 3
        # =====================================================

        x, _ = self.lstm3(x)

        x = x[:, -1, :]

        x = self.dropout3(x)

        x = self.bn3(x)

        # =====================================================
        # Dense
        # =====================================================

        x = self.fc1(x)

        x = self.relu(x)

        x = self.dropout4(x)

        x = self.fc2(x)

        x = self.relu(x)

        x = self.output(x)

        x = self.sigmoid(x)

        return x

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn

from torch.utils.data import (
    TensorDataset,
    DataLoader
)

from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    balanced_accuracy_score,
    precision_recall_curve
)


# =========================================================
# CONFIG
# =========================================================

DATA_PATH = "C:/Nguyen Tri/Code/Statisanalyss/all_training_data.csv"

SEQ_LENGTH = 20

BATCH_SIZE = 64

EPOCHS = 50

LEARNING_RATE = 1e-4

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print("Using device:", DEVICE)


# =========================================================
# LOAD DATA
# =========================================================

df = pd.read_csv(DATA_PATH)


# =========================================================
# REMOVE LEAKAGE
# =========================================================

drop_cols = [

    'PitInTime',
    'PitOutTime',

    'LapTime',
    'Sector1Time',
    'Sector2Time',
    'Sector3Time',

    'SessionTime',
    'Time',

    'Deleted',
    'DeletedReason',

    'FastF1Generated',
    'IsAccurate'
]

existing_drop_cols = [

    col for col in drop_cols
    if col in df.columns
]

df = df.drop(
    columns=existing_drop_cols,
    errors='ignore'
)


# =========================================================
# TRAIN TEST SPLIT
# =========================================================

if 2025 in df['Year'].values:

    train_df = df[
        df['Year'] < 2025
    ].copy()

    test_df = df[
        df['Year'] == 2025
    ].copy()

else:

    latest_year = df['Year'].max()

    train_df = df[
        df['Year'] < latest_year
    ].copy()

    test_df = df[
        df['Year'] == latest_year
    ].copy()


# =========================================================
# FIX HISTORICAL PIT LEAKAGE
# =========================================================

historical_pit = (

    train_df[
        train_df['HasPitStop'] == 1
    ]

    .groupby(
        ['Compound', 'Team']
    )['LapNumber']

    .mean()

    .reset_index()
)

historical_pit.columns = [

    'Compound',
    'Team',
    'historical_pit_lap'
]

train_df = train_df.drop(
    columns=['historical_pit_lap'],
    errors='ignore'
)

test_df = test_df.drop(
    columns=['historical_pit_lap'],
    errors='ignore'
)

train_df = train_df.merge(
    historical_pit,
    on=['Compound', 'Team'],
    how='left'
)

test_df = test_df.merge(
    historical_pit,
    on=['Compound', 'Team'],
    how='left'
)

global_pit_mean = (
    historical_pit[
        'historical_pit_lap'
    ].mean()
)

train_df['historical_pit_lap'] = (
    train_df['historical_pit_lap']
    .fillna(global_pit_mean)
)

test_df['historical_pit_lap'] = (
    test_df['historical_pit_lap']
    .fillna(global_pit_mean)
)

train_df['pit_window_delta'] = (
    train_df['LapNumber']
    -
    train_df['historical_pit_lap']
)

test_df['pit_window_delta'] = (
    test_df['LapNumber']
    -
    test_df['historical_pit_lap']
)


# =========================================================
# FEATURES
# =========================================================

categorical_cols = [
    'Compound',
    'Team'
]

continuous_cols = [

    # race state
    'LapTime_Seconds',
    'Position',
    'LapNumber',
    'TyreLife',
    'TrackStatus',

    # pace evolution
    'delta_laptime',
    'CumulativeTimeStint',
    'race_progress_fraction',

    # tire degradation
    'relative_tire_age',
    'tire_compound_age_delta',
    'tire_performance_decay',

    # rolling pace metrics
    'rolling_pace_mean_5',
    'pace_std_5',
    'pace_degradation_slope',

    # pit strategy features
    'historical_pit_lap',
    'pit_window_delta',

    # telemetry
    'Speed_mean',
    'Speed_max',
    'Speed_std',

    'RPM_mean',
    'RPM_max',

    'Throttle_mean',
    'Throttle_std',

    'Brake_mean',
    'Brake_sum',

    'DRS_mean',
    'DRS_sum',

    # traffic
    'DistanceToDriverAhead_mean',
    'DistanceToDriverAhead_min',

    # gearbox
    'nGear_mean',
    'nGear_max',

    # engineered strategy indicators
    'traffic_pressure',
    'drs_dependency',
    'thermal_stress_proxy',
    'high_speed_stress',
    'brake_aggression',
    'pace_vs_ahead'
]

# =========================================================
# ENCODE CATEGORICALS
# =========================================================

combined_df = pd.concat(
    [train_df, test_df],
    axis=0
)

combined_df = pd.get_dummies(
    combined_df,
    columns=categorical_cols,
    drop_first=False
)

train_df = combined_df[
    combined_df['Year'] < 2025
].copy()

test_df = combined_df[
    combined_df['Year'] == 2025
].copy()

encoded_cat_cols = [

    col for col in combined_df.columns

    if (
        col.startswith("Compound_")
        or
        col.startswith("Team_")
    )
]

feature_cols = (
    continuous_cols +
    encoded_cat_cols
)


# =========================================================
# NORMALIZATION
# =========================================================

scaler = StandardScaler()

train_df[continuous_cols] = (
    scaler.fit_transform(
        train_df[continuous_cols]
    )
)

test_df[continuous_cols] = (
    scaler.transform(
        test_df[continuous_cols]
    )
)

train_df = train_df.fillna(0)
test_df = test_df.fillna(0)


# =========================================================
# SEQUENCE CREATION
# =========================================================

def create_driver_sequences(
    data,
    feature_columns,
    target_column='HasPitStop',
    seq_length=20
):

    X = []
    y = []

    grouped = data.groupby(
        ['RaceID', 'DriverNumber']
    )

    for _, group in grouped:

        group = group.sort_values(
            'LapNumber'
        )

        feat = group[
            feature_columns
        ].values

        targ = group[
            target_column
        ].values

        if len(group) < seq_length:
            continue

        for i in range(
            len(group) - seq_length + 1
        ):

            X.append(
                feat[
                    i:i+seq_length
                ]
            )

            y.append(
                targ[
                    i+seq_length-1
                ]
            )

    return (

        np.array(
            X,
            dtype=np.float32
        ),

        np.array(
            y,
            dtype=np.float32
        )
    )


X_train, y_train = create_driver_sequences(
    train_df,
    feature_cols,
    seq_length=SEQ_LENGTH
)

X_test, y_test = create_driver_sequences(
    test_df,
    feature_cols,
    seq_length=SEQ_LENGTH
)

print(X_train.shape)
print(X_test.shape)


# =========================================================
# PYTORCH DATASET
# =========================================================

X_train_tensor = torch.tensor(
    X_train,
    dtype=torch.float32
)

y_train_tensor = torch.tensor(
    y_train,
    dtype=torch.float32
).unsqueeze(1)

X_test_tensor = torch.tensor(
    X_test,
    dtype=torch.float32
)

y_test_tensor = torch.tensor(
    y_test,
    dtype=torch.float32
).unsqueeze(1)

train_dataset = TensorDataset(
    X_train_tensor,
    y_train_tensor
)

test_dataset = TensorDataset(
    X_test_tensor,
    y_test_tensor
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=256,
    shuffle=False
)


# =========================================================
# MODEL
# =========================================================

class F1PitStopPredictor(nn.Module):

    def __init__(
        self,
        input_size
    ):

        super().__init__()

        self.lstm1 = nn.LSTM(
            input_size=input_size,
            hidden_size=256,
            batch_first=True,
            bidirectional=True
        )

        self.bn1 = nn.BatchNorm1d(
            512
        )

        self.dropout1 = nn.Dropout(
            0.2
        )

        self.lstm2 = nn.LSTM(
            input_size=512,
            hidden_size=128,
            batch_first=True,
            bidirectional=True
        )

        self.bn2 = nn.BatchNorm1d(
            256
        )

        self.dropout2 = nn.Dropout(
            0.3
        )

        self.lstm3 = nn.LSTM(
            input_size=256,
            hidden_size=64,
            batch_first=True,
            bidirectional=True
        )

        self.bn3 = nn.BatchNorm1d(
            128
        )

        self.dropout3 = nn.Dropout(
            0.3
        )

        self.fc1 = nn.Linear(
            128,
            64
        )

        self.dropout4 = nn.Dropout(
            0.2
        )

        self.fc2 = nn.Linear(
            64,
            32
        )

        self.output = nn.Linear(
            32,
            1
        )

        self.relu = nn.ReLU()

    def forward(
        self,
        x
    ):

        x, _ = self.lstm1(x)

        x = self.dropout1(x)

        x = x.permute(0, 2, 1)
        x = self.bn1(x)
        x = x.permute(0, 2, 1)

        x, _ = self.lstm2(x)

        x = self.dropout2(x)

        x = x.permute(0, 2, 1)
        x = self.bn2(x)
        x = x.permute(0, 2, 1)

        x, _ = self.lstm3(x)

        x = x[:, -1, :]

        x = self.dropout3(x)

        x = self.bn3(x)

        x = self.fc1(x)

        x = self.relu(x)

        x = self.dropout4(x)

        x = self.fc2(x)

        x = self.relu(x)

        x = self.output(x)

        return x


model = F1PitStopPredictor(
    input_size=X_train.shape[2]
).to(DEVICE)


# =========================================================
# LOSS
# =========================================================

positive_weight = (
    len(y_train[y_train == 0])
    /
    len(y_train[y_train == 1])
)

positive_weight_tensor = torch.tensor(
    [positive_weight], 
    dtype=torch.float32
).to(DEVICE)

criterion = nn.BCEWithLogitsLoss(pos_weight=positive_weight_tensor)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='max',
    factor=0.5,
    patience=4
)


# =========================================================
# TRAINING LOOP
# =========================================================

best_auc_pr = 0

patience = 8

counter = 0

for epoch in range(EPOCHS):

    model.train()

    total_loss = 0

    for batch_x, batch_y in train_loader:

        batch_x = batch_x.to(DEVICE)

        batch_y = batch_y.to(DEVICE)

        optimizer.zero_grad()

        outputs = model(batch_x)

        loss = criterion(
            outputs,
            batch_y
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            1.0
        )

        optimizer.step()

        total_loss += loss.item()

    # =====================================================
    # VALIDATION
    # =====================================================

    model.eval()

    all_probs = []

    all_targets = []

    with torch.no_grad():

        for batch_x, batch_y in test_loader:

            batch_x = batch_x.to(DEVICE)

            outputs = model(batch_x)

            probs = (
                torch.sigmoid(outputs)
                .squeeze()
                .cpu()
                .numpy()
            )

            all_probs.extend(probs)

            all_targets.extend(
                batch_y.numpy().flatten()
            )

    auc_pr = average_precision_score(
        all_targets,
        all_probs
    )

    roc_auc = roc_auc_score(
        all_targets,
        all_probs
    )

    precision = precision_score(
        all_targets,
        (np.array(all_probs) >= 0.5).astype(int)
    )

    recall = recall_score(
        all_targets,
        (np.array(all_probs) >= 0.5).astype(int)
    )

    f1 = f1_score(
        all_targets,
        (np.array(all_probs) >= 0.5).astype(int)
    )

    scheduler.step(auc_pr)

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"Loss: {total_loss/len(train_loader):.4f} | "
        f"ROC-AUC: {roc_auc:.4f} | "
        f"AUC-PR: {auc_pr:.4f} | "
        f"Precision: {precision:.4f} | "
        f"Recall: {recall:.4f} | "
        f"F1-Score: {f1:.4f}"
    )

    if auc_pr > best_auc_pr:

        best_auc_pr = auc_pr

        counter = 0

        torch.save(
            model.state_dict(),
            "best_f1_model.pt"
        )

# =========================================================
# LOAD BEST MODEL
# =========================================================

# model.load_state_dict(
#     torch.load(
#         "best_f1_model.pt"
#     )
# )


# =========================================================
# FINAL EVALUATION
# =========================================================

model.eval()

all_probs = []

all_targets = []

with torch.no_grad():

    for batch_x, batch_y in test_loader:

        batch_x = batch_x.to(DEVICE)

        outputs = model(batch_x)

        probs = (
            torch.sigmoid(outputs)
            .squeeze()
            .cpu()
            .numpy()
        )

        all_probs.extend(probs)

        all_targets.extend(
            batch_y.numpy().flatten()
        )

y_pred_probs = np.array(
    all_probs
)

y_test_np = np.array(
    all_targets
)


# =========================================================
# BEST THRESHOLD
# =========================================================

precision_vals, recall_vals, thresholds = (
    precision_recall_curve(
        y_test_np,
        y_pred_probs
    )
)

f1_scores = (

    2
    *
    precision_vals
    *
    recall_vals

    /

    (
        precision_vals
        +
        recall_vals
        +
        1e-8
    )
)

best_idx = np.argmax(
    f1_scores[:-1]
)

best_threshold = thresholds[
    best_idx
]

print(
    f"Best Threshold: "
    f"{best_threshold:.4f}"
)


# =========================================================
# FINAL PREDICTIONS
# =========================================================

y_pred = (
    y_pred_probs >= best_threshold
).astype(int)

precision = precision_score(
    y_test_np,
    y_pred
)

recall = recall_score(
    y_test_np,
    y_pred
)

f1 = f1_score(
    y_test_np,
    y_pred
)

roc_auc = roc_auc_score(
    y_test_np
    y_pred_probs
)

auc_pr = average_precision_score(
    y_test_np,
    y_pred_probs
)

balanced_acc = balanced_accuracy_score(
    y_test_np,
    y_pred
)

tn, fp, fn, tp = confusion_matrix(
    y_test_np,
    y_pred
).ravel()

specificity = tn / (tn + fp)


# =========================================================
# RESULTS
# =========================================================

print("\n--- Evaluation Metrics ---")

print(f"Precision:         {precision:.4f}")
print(f"Recall:            {recall:.4f}")
print(f"F1-Score:          {f1:.4f}")
print(f"Specificity:       {specificity:.4f}")
print(f"Balanced Accuracy: {balanced_acc:.4f}")
print(f"ROC-AUC:           {roc_auc:.4f}")
print(f"AUC-PR:            {auc_pr:.4f}")

print("\n--- Confusion Matrix ---")

print(f"True Negatives (TN):  {tn}")
print(f"False Positives (FP): {fp}")
print(f"False Negatives (FN): {fn}")
print(f"True Positives (TP):  {tp}")


# =========================================================
# SAVE MODEL
# =========================================================

torch.save(
    model.state_dict(),
    "f1_bilstm_model.pt"
)

print("\nModel Saved")

Using device: cpu
(111954, 20, 59)
(14833, 20, 59)
Epoch 1/50 | Loss: 0.8713 | ROC-AUC: 0.9871 | AUC-PR: 0.8101 | Precision: 0.7251 | Recall: 0.7444 | F1-Score: 0.7346
Epoch 2/50 | Loss: 0.6513 | ROC-AUC: 0.9888 | AUC-PR: 0.8516 | Precision: 0.7952 | Recall: 0.7422 | F1-Score: 0.7678
Epoch 3/50 | Loss: 0.6119 | ROC-AUC: 0.9916 | AUC-PR: 0.8813 | Precision: 0.7283 | Recall: 0.8400 | F1-Score: 0.7802
Epoch 4/50 | Loss: 0.5218 | ROC-AUC: 0.9921 | AUC-PR: 0.8793 | Precision: 0.7800 | Recall: 0.7956 | F1-Score: 0.7877
Epoch 5/50 | Loss: 0.4954 | ROC-AUC: 0.9915 | AUC-PR: 0.8857 | Precision: 0.7633 | Recall: 0.8311 | F1-Score: 0.7957
Epoch 6/50 | Loss: 0.4519 | ROC-AUC: 0.9891 | AUC-PR: 0.8698 | Precision: 0.8234 | Recall: 0.7356 | F1-Score: 0.7770
Epoch 7/50 | Loss: 0.4172 | ROC-AUC: 0.9919 | AUC-PR: 0.8892 | Precision: 0.8486 | Recall: 0.7600 | F1-Score: 0.8019
Epoch 8/50 | Loss: 0.4001 | ROC-AUC: 0.9907 | AUC-PR: 0.8948 | Precision: 0.8157 | Recall: 0.8067 | F1-Score: 0.8112
Epoch 9/50 | 

In [11]:
def run_validation_and_get_probabilities(
    model,
    test_loader,
    race_ids,
    driver_numbers,
    lap_numbers
):

    model.eval()
    all_probs = []

    with torch.no_grad():

        for batch_x, _ in test_loader:

            batch_x = batch_x.to(DEVICE)

            outputs = model(batch_x)

            probs = (
                outputs
                .squeeze()
                .cpu()
                .numpy()
            )

            all_probs.extend(probs)

    # Create a DataFrame with the probabilities and metadata
    prob_df = pd.DataFrame({
        'RaceID': race_ids,
        'DriverNumber': driver_numbers,
        'LapNumber': lap_numbers,
        'PitStopProbability': all_probs
    })

    return prob_df

In [ ]:
# Load the best model's state dictionary
model.load_state_dict(torch.load("best_f1_model.pt"))

df = pd.read_csv("/content/drive/Othercomputers/My Computer/Nguyen Tri/Code/Statisanalyss/all_training_data.csv")

# Get the pit stop probabilities for each lap in the test set
pit_probabilities_df = run_validation_and_get_probabilities(
    model,
    test_loader,
    race_ids_test,
    driver_numbers_test,
    lap_numbers_test
)

# Display the DataFrame with pit stop probabilities
display(pit_probabilities_df.head())

,RaceID,DriverNumber,LapNumber,PitStopProbability
0,Abu Dhabi Grand Prix,1,20,-8.281097
1,Abu Dhabi Grand Prix,1,21,-8.574289
2,Abu Dhabi Grand Prix,1,22,-8.104769
3,Abu Dhabi Grand Prix,1,23,4.851282
4,Abu Dhabi Grand Prix,1,24,-12.584503


In [17]:
# Load the original un-normalized dataset
csv_path = "C:/Nguyen Tri/Code/Statisanalyss/all_training_data.csv"
full_df = pd.read_csv(csv_path)

# Merge the probabilities back into the full dataset
# We join on RaceID, DriverNumber, and LapNumber to ensure everything aligns perfectly
full_df = full_df.merge(
    pit_probabilities_all_df[['RaceID', 'DriverNumber', 'LapNumber', 'PitStopProbability']], 
    on=['RaceID', 'DriverNumber', 'LapNumber'], 
    how='left'
)

# Fill the first 19 laps (which don't have predictions due to our sequence length) with 0.0
full_df['PitStopProbability'] = full_df['PitStopProbability'].fillna(0.0)

# Save the updated data back to the CSV
full_df.to_csv(csv_path, index=False)

print("Successfully added 'PitStopProbability' to all_training_data.csv!")


Successfully added 'PitStopProbability' to all_training_data.csv!
